In [ ]:
import xgboost as xgt

In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split,KFold,RandomizedSearchCV,cross_val_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from sklearn.preprocessing import StandardScaler,OrdinalEncoder
import mlflow

In [ ]:
df=pd.read_csv("D:/powerbi/medical_insurance.csv")
df.columns

Index(['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges'], dtype='object')

In [ ]:
x=df[['age', 'sex', 'bmi', 'children', 'smoker']].copy()
y=df["charges"]

In [ ]:
x_num=x.select_dtypes(include="number")
x_str=x.select_dtypes(exclude="number")

In [ ]:
preprocess=ColumnTransformer(
    transformers=[
        ("numeric values",StandardScaler(),x_num.columns),
        ("string values",OrdinalEncoder(),x_str.columns)
    ],
    remainder="passthrough"
)

In [ ]:
pipeline=Pipeline(
    steps=[
        ("preprocess",preprocess),
        ("model",xgt.XGBRegressor())
    ]
)

In [ ]:
param={
    "model__max_depth":[20,30,40,50],
    "model__n_estimators":[100,150,200],
    "model__learning_rate":[0.1,0.2,0.5],
}

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42,test_size=0.2)
cross=KFold(n_splits=5,shuffle=True,random_state=42)

In [ ]:
model=RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param,
    cv=cross,
    scoring="r2",
    n_iter=10
)

In [45]:
model.fit(x_train,y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'model__learning_rate': [0.1, 0.2, ...], 'model__max_depth': [20, 30, ...], 'model__n_estimators': [100, 150, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchange

In [46]:
y_pred=model.predict(x_test)
r2=(r2_score(y_test,y_pred))
mae=mean_absolute_error(y_test,y_pred)
mse=mean_squared_error(y_test,y_pred)

In [53]:
best_model=model.best_estimator_
best_model=best_model.named_steps["model"]


In [54]:
# If best_model is a Pipeline, extract the final step
  # replace "xgbregressor" with the step name you used

mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("Medical_Insurance_cost_prediction")

with mlflow.start_run(run_name="XGboost Classifier"):
    mlflow.log_params(model.best_params_)
    mlflow.log_metric("R2", r2)
    mlflow.log_metric("MSE", mse)
    mlflow.log_metric("MAE", mae)

    # Log the actual XGBoost model, not the pipeline
    mlflow.xgboost.log_model(best_model, "Xgboost_classifier")

2026/03/31 13:18:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGboost Classifier at: http://127.0.0.1:5000/#/experiments/1/runs/e7b6c521ec91442a8add4f5df0c0c800
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
